In [5]:
import pandas as pd 
import numpy as np 
import plotly.express as px 
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import BorderlineSMOTE
import optuna
import joblib 
import streamlit as st 
import joblib


In [6]:
df=pd.read_csv("credit_risk_dataset.csv")
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


#  Dataset Column Description – Credit Risk Data

##  Column Descriptions

| Column | Description |
|--------|------------|
| person_age | Borrower's age in years. |
| person_income | Annual income of the borrower. |
| person_home_ownership | Home ownership status of the borrower (RENT / OWN / MORTGAGE / OTHER). |
| person_emp_length | Employment length — number of years the borrower has been employed. |
| loan_intent | Purpose of the loan (EDUCATION, MEDICAL, VENTURE, PERSONAL, HOMEIMPROVEMENT, DEBTCONSOLIDATION). |
| loan_grade | Credit grade assigned to the loan, from A (lowest risk) to G (highest risk). |
| loan_amnt | Amount of money requested for the loan. |
| loan_int_rate | Interest rate of the loan (in percent). |
| loan_status | Indicates whether the borrower defaulted on the loan (1 = Default, 0 = Non-default). |
| loan_percent_income | Ratio of the loan amount to the borrower's annual income. |
| cb_person_default_on_file |(Credit Bureau) Indicates whether the borrower has a history of default in the credit bureau records (Y/N). |
| cb_person_cred_hist_length | Length of the borrower's credit history (in years). |

# Data Review 

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.5 MB


In [8]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [9]:
df.columns

Index(['person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_status', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length'],
      dtype='str')

In [10]:
df.isna().sum().sort_values(ascending=False)

loan_int_rate                 3116
person_emp_length              895
person_income                    0
person_age                       0
person_home_ownership            0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

# Remove Duplicates

In [11]:
df.duplicated().sum()

np.int64(165)

In [12]:
df=df.drop_duplicates()

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.isna().sum().sort_values(ascending=False)

loan_int_rate                 3095
person_emp_length              887
person_income                    0
person_age                       0
person_home_ownership            0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

# Feature engineering

In [15]:
df["loan_to_income"] = df["loan_amnt"] / df["person_income"]

In [16]:
df.loc[1:5, ["loan_amnt", "person_income", "loan_percent_income", "loan_to_income", "loan_status"]]

,loan_amnt,person_income,loan_percent_income,loan_to_income,loan_status
1,1000,9600,0.10,0.104167,0
2,5500,9600,0.57,0.572917,1
3,35000,65500,0.53,0.534351,1
4,35000,54400,0.55,0.643382,1
5,2500,9900,0.25,0.252525,1


In [17]:
wrong = df[(df["loan_to_income"] - df["loan_percent_income"]).abs() > 0.01]
print(f"Wrong rows: {len(wrong)}")


Wrong rows: 387


In [18]:
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_to_income
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3,0.593220
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2,0.104167
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3,0.572917
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2,0.534351
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4,0.643382


# Describing Data

In [19]:
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length,loan_to_income
count,32416.000000,3.241600e+04,31529.00000,32416.000000,29321.000000,32416.000000,32416.000000,32416.000000,32416.000000
mean,27.747008,6.609164e+04,4.79051,9593.845632,11.017265,0.218688,0.170250,5.811297,0.170599
std,6.354100,6.201558e+04,4.14549,6322.730241,3.241680,0.413363,0.106812,4.059030,0.107081
min,20.000000,4.000000e+03,0.00000,500.000000,5.420000,0.000000,0.000000,2.000000,0.000789
25%,23.000000,3.854200e+04,2.00000,5000.000000,7.900000,0.000000,0.090000,3.000000,0.089711
50%,26.000000,5.500000e+04,4.00000,8000.000000,10.990000,0.000000,0.150000,4.000000,0.148148
75%,30.000000,7.921800e+04,7.00000,12250.000000,13.470000,0.000000,0.230000,8.000000,0.229167
max,144.000000,6.000000e+06,123.00000,35000.000000,23.220000,1.000000,0.830000,30.000000,0.830000


In [20]:
df[df["person_age"]>=120].count()

person_age                    5
person_income                 5
person_home_ownership         5
person_emp_length             5
loan_intent                   5
loan_grade                    5
loan_amnt                     5
loan_int_rate                 4
loan_status                   5
loan_percent_income           5
cb_person_default_on_file     5
cb_person_cred_hist_length    5
loan_to_income                5
dtype: int64

In [21]:
df=df[df["person_age"]<=120]

In [22]:
df[df["person_age"]>=120].count()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
loan_to_income                0
dtype: int64

In [23]:
df[df['person_emp_length'] > df['person_age']]

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_to_income
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3,0.593220
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4,0.104167


In [24]:
df = df[(df["person_emp_length"] <= df["person_age"]) | (df["person_emp_length"].isna())]

In [25]:
df[df['person_emp_length'] > df['person_age']]

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_to_income


# Unique Values

In [26]:
for col in df.select_dtypes(exclude="number").columns:
    print(col)
    print(f"   nunique: {df[col].nunique()}")
    print(f"   unique:  {df[col].unique().tolist()}")
    print("-" * 50)

person_home_ownership
   nunique: 4
   unique:  ['OWN', 'MORTGAGE', 'RENT', 'OTHER']
--------------------------------------------------
loan_intent
   nunique: 6
   unique:  ['EDUCATION', 'MEDICAL', 'VENTURE', 'PERSONAL', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION']
--------------------------------------------------
loan_grade
   nunique: 7
   unique:  ['B', 'C', 'A', 'D', 'E', 'F', 'G']
--------------------------------------------------
cb_person_default_on_file
   nunique: 2
   unique:  ['N', 'Y']
--------------------------------------------------


# Detect Outliers

In [27]:
from datasist.structdata import detect_outliers

for col in df.select_dtypes(include="number").columns.drop("loan_status"):
    outliers = detect_outliers(data=df.dropna(subset=[col]), n=0, features=[col])

    print(f" {col}:")
    print(f"    Outliers Count: {len(outliers)}")
    print(f"    Outliers Percentage: {len(outliers) / len(df) * 100:.2f}%")
    print("-" * 50)

 person_age:
    Outliers Count: 1486
    Outliers Percentage: 4.59%
--------------------------------------------------
 person_income:
    Outliers Count: 1474
    Outliers Percentage: 4.55%
--------------------------------------------------
 person_emp_length:
    Outliers Count: 850
    Outliers Percentage: 2.62%
--------------------------------------------------
 loan_amnt:
    Outliers Count: 1678
    Outliers Percentage: 5.18%
--------------------------------------------------
 loan_int_rate:
    Outliers Count: 6
    Outliers Percentage: 0.02%
--------------------------------------------------
 loan_percent_income:
    Outliers Count: 649
    Outliers Percentage: 2.00%
--------------------------------------------------
 cb_person_cred_hist_length:
    Outliers Count: 1138
    Outliers Percentage: 3.51%
--------------------------------------------------
 loan_to_income:
    Outliers Count: 732
    Outliers Percentage: 2.26%
--------------------------------------------------


In [28]:
df = df.reset_index(drop=True)

In [29]:
df.to_csv("credit_risk_clean.csv", index=False)

# EDA for Cat


In [30]:
for col in ["loan_grade", "loan_intent", "person_home_ownership", "cb_person_default_on_file"]:
    fig = px.histogram(df, x=col, y="loan_status", histfunc="avg", text_auto=".1%",
                       title=f"Failure to Pay Rate in {col}", color_discrete_sequence=["#E19E57"],
                       category_orders={"loan_grade": ["A", "B", "C", "D", "E", "F", "G"]})
    fig.update_layout(yaxis_title="Failure to Pay Rate %", xaxis_title=col, yaxis_tickformat=".0%")
    fig.show()

 # chart between interest_rate and grades

In [31]:
fig_3 = px.histogram(df, x="loan_grade", y="loan_int_rate",
                     histfunc="avg", text_auto=".2f",color_discrete_sequence=["#21997B"],
                     category_orders={"loan_grade": ["A", "B", "C", "D", "E", "F", "G"]})
fig_3

# This chart shows that the interest rate goes up with every grade

## EDA Summary for Categorical Features
- **loan_grade:** The bank gives each loan a grade from A (safe) to G (risky). Only 10% of grade A borrowers failed to pay, compared to 98% in grade G. The risk jumps sharply at grade D (from 21% in C to 59% in D).
- **home_ownership:** Renters have the highest risk (32%), while homeowners have the lowest risk (7.5%). This is likely because renters pay rent every month, while homeowners are more financially stable.
- **loan_intent:** Debt consolidation (28.7%) and medical loans (26.8%) are the riskiest, likely because these borrowers are already under financial pressure. Venture (14.9%) and education loans (17.3%) are the safest.
- **cb_person_default_on_file:** Borrowers who defaulted before fail to pay 37.9% of the time, compared to 18.4% for others. All borrowers in grades A and B have never defaulted before .

In [32]:
df[df["cb_person_default_on_file"] == "Y"]["loan_grade"].value_counts()

loan_grade
C    3245
D    1872
E     464
F     112
G      36
Name: count, dtype: int64

# EDA for Num

In [33]:
for col in ["loan_to_income", "loan_int_rate", "person_income", "loan_amnt",
            "person_emp_length", "person_age", "cb_person_cred_hist_length"]:
    data = df.dropna(subset=[col])
    data = data[data[col] <= data[col].quantile(0.99)]
    fig_2 = px.histogram(data, x=col, y="loan_status", histfunc="avg", nbins=10, text_auto=".1%",
                         title=f"Failure to Pay Rate in {col}", color_discrete_sequence=["#0DABE4"])
    fig_2.update_layout(yaxis_title="Failure to Pay Rate", yaxis_tickformat=".0%", bargap=0.1)
    fig_2.show()

## EDA Summary for Numerical Features
- **loan_to_income:** The failure rate rises sharply when the loan is more than about 30% of income.
- **loan_int_rate:** Higher interest rates come with a higher failure rate.
- **person_income:** Lower income means higher risk, and the failure rate falls as income goes up.
- **age / credit history length:** Almost no effect. 
- **Hidden pattern:** Even borrowers with grade A or B have a 62% failure rate if the loan is more than 30% of their income, compared to only 7% when it is below 30% (see the code below).

In [34]:
ab = df[df["loan_grade"].isin(["A", "B"])]
ab.groupby(ab["loan_to_income"] > 0.3)["loan_status"].mean()*100

loan_to_income
False     6.858029
True     62.153070
Name: loan_status, dtype: float64

# Split Data

In [35]:
x=df.drop(columns="loan_status")
y=df["loan_status"]


In [36]:
num=df.select_dtypes(include="number").columns.drop("loan_status").to_list()
cat=df.select_dtypes(exclude="number").columns.drop("loan_grade").to_list()
order_cat=["loan_grade"]
category_orders=[["A", "B", "C", "D", "E", "F", "G"]]

print(f"numeric_col:{num}")

print(f"categorical_col:{cat}")

print(f"ordered:{order_cat}")

print(f"category_orders: {category_orders}")


numeric_col:['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'loan_to_income']
categorical_col:['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
ordered:['loan_grade']
category_orders: [['A', 'B', 'C', 'D', 'E', 'F', 'G']]


## Check Imbalnce

In [37]:
(y.value_counts(normalize=True)*100).round(2)


loan_status
0    78.13
1    21.87
Name: proportion, dtype: float64

In [38]:
y_imbalance = (df["loan_status"].value_counts(normalize=True) * 100 ).round(2).reset_index()
y_imbalance.columns = ["loan_status", "percentage"]
y_imbalance["loan_status"] = y_imbalance["loan_status"].map({0: "Paid", 1: "Failed to Pay"})
fig_1=px.bar(y_imbalance,x="loan_status",y="percentage",text="percentage",title="Loan Distribution",
             color="loan_status",color_discrete_map={"Paid": "#4FA19D", "Failed to Pay": "#7C57E1"})
fig_1.show()

# Preprocessing

_________________________________

In [39]:
df.isna().sum().sort_values(ascending=False)

loan_int_rate                 3094
person_emp_length              887
person_age                       0
person_home_ownership            0
person_income                    0
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
loan_to_income                   0
dtype: int64

In [40]:
(df.isna().mean()*100).round(2).sort_values(ascending=False)

loan_int_rate                 9.55
person_emp_length             2.74
person_age                    0.00
person_home_ownership         0.00
person_income                 0.00
loan_intent                   0.00
loan_grade                    0.00
loan_amnt                     0.00
loan_status                   0.00
loan_percent_income           0.00
cb_person_default_on_file     0.00
cb_person_cred_hist_length    0.00
loan_to_income                0.00
dtype: float64

# Num pipeline

In [41]:
from sklearn.impute import KNNImputer
knn=KNNImputer(n_neighbors=5)
from sklearn.preprocessing import RobustScaler
robust=RobustScaler()
num_pipline=Pipeline(steps=[("robust",robust),("knn",knn)])
display(num_pipline)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('robust', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"missing_values missing_values: int, float, str, np.nan or None, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`should be set to np.nan, since `pd.NA` will be converted to np.nan.",nan
,"n_neighbors n_neighbors: int, default=5Number of neighboring samples to use for imputation.",5


# Ohe pipeline

In [42]:
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False,drop="first",handle_unknown="ignore")
ohe_pipeline=Pipeline(steps=[("ohe",ohe)])
display(ohe_pipeline)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('ohe', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category

# Ordinal pipeline

In [43]:
from  sklearn.preprocessing import OrdinalEncoder
ord=OrdinalEncoder(categories=category_orders)
ordinal_pipeline=Pipeline(steps=[("ord",ord)])
display(ordinal_pipeline)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('ord', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.","[['A', 'B', ...]]"
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'error'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",None
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of output categories for each inputfeature when considering infrequ

# Column transformer

In [44]:
from sklearn.compose import ColumnTransformer
preprocessing=ColumnTransformer(transformers=[("num",num_pipline,num),
                                              ("ohe",ohe,cat),
                                              ("ord",ord,order_cat)
                                              ],remainder="passthrough",verbose_feature_names_out=False)
display(preprocessing)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per tran


# Models_score & cross_validate

In [45]:
from sklearn.model_selection import cross_validate, StratifiedKFold  
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
 

models = [
    ("logistic_regression", LogisticRegression(random_state=42)),
    ("knn_classifier", KNeighborsClassifier()),
    ("decision_tree", DecisionTreeClassifier(random_state=42)),
    ("random_forest", RandomForestClassifier(random_state=42)),
    ("xgboost_classifier", XGBClassifier(random_state=42)),
    ("catboost_classifier", CatBoostClassifier(verbose=0, random_state=42)),
    ("lightgbm_classifier", LGBMClassifier(verbose=-1, random_state=42))]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)




for model in models:
    model_pipeline=Pipeline(steps=[("preprocessing",preprocessing),("model",model[1])])
  
    result=cross_validate(model_pipeline,x,y,cv=cv,scoring="f1",return_train_score=True,n_jobs=1)


    print(model[0])
    print(f"Train_score: {result["train_score"].mean():.2f}")
    print(f"Test_score:  {result["test_score"].mean():.2f}")
    print("-" * 50)



logistic_regression
Train_score: 0.62
Test_score:  0.62
--------------------------------------------------
knn_classifier
Train_score: 0.79
Test_score:  0.72
--------------------------------------------------
decision_tree
Train_score: 1.00
Test_score:  0.76
--------------------------------------------------
random_forest
Train_score: 1.00
Test_score:  0.83
--------------------------------------------------
xgboost_classifier
Train_score: 0.90
Test_score:  0.84
--------------------------------------------------
catboost_classifier
Train_score: 0.87
Test_score:  0.84
--------------------------------------------------
lightgbm_classifier
Train_score: 0.85
Test_score:  0.83
--------------------------------------------------


# BorderlineSMOTE


In [46]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline  
 
models = [
    ("logistic_regression", LogisticRegression(random_state=42)),
    ("knn_classifier", KNeighborsClassifier()),
    ("decision_tree", DecisionTreeClassifier(random_state=42)),
    ("random_forest", RandomForestClassifier(random_state=42)),
    ("xgboost_classifier", XGBClassifier(random_state=42)),
    ("catboost_classifier", CatBoostClassifier(verbose=0, random_state=42)),
    ("lightgbm_classifier", LGBMClassifier(verbose=-1, random_state=42))]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



for model in models:
    model_pipeline_Borderline=Pipeline(steps=[("preprocessing",preprocessing),
                                   ("smote",BorderlineSMOTE(random_state=42)),
                                   ("model",model[1])])
    

    result=cross_validate(model_pipeline_Borderline,x,y,cv=cv,scoring="f1",return_train_score=True,n_jobs=1)
    

    print(model[0])
    print(f"Train_score: {result["train_score"].mean():.2f}")
    print(f"Test_score:  {result["test_score"].mean():.2f}")
    print("-" * 50)




logistic_regression
Train_score: 0.60
Test_score:  0.60
--------------------------------------------------
knn_classifier
Train_score: 0.78
Test_score:  0.63
--------------------------------------------------
decision_tree
Train_score: 1.00
Test_score:  0.72
--------------------------------------------------
random_forest
Train_score: 1.00
Test_score:  0.82
--------------------------------------------------
xgboost_classifier
Train_score: 0.89
Test_score:  0.83
--------------------------------------------------
catboost_classifier
Train_score: 0.87
Test_score:  0.84
--------------------------------------------------
lightgbm_classifier
Train_score: 0.84
Test_score:  0.83
--------------------------------------------------


# SMOTE

In [47]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline  
from imblearn.over_sampling import SMOTE
 
models = [
    ("logistic_regression", LogisticRegression(random_state=42)),
    ("knn_classifier", KNeighborsClassifier()),
    ("decision_tree", DecisionTreeClassifier(random_state=42)),
    ("random_forest", RandomForestClassifier(random_state=42)),
    ("xgboost_classifier", XGBClassifier(random_state=42)),
    ("catboost_classifier", CatBoostClassifier(verbose=0, random_state=42)),
    ("lightgbm_classifier", LGBMClassifier(verbose=-1, random_state=42))]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



for model in models:
    model_pipeline_smote=Pipeline(steps=[("preprocessing",preprocessing),
                                   ("smote",SMOTE(random_state=42)),
                                   ("model",model[1])])
    

    result=cross_validate(model_pipeline_smote,x,y,cv=cv,scoring="f1",return_train_score=True,n_jobs=1)
    

    print(model[0])
    print(f"Train_score: {result["train_score"].mean():.2f}")
    print(f"Test_score:  {result["test_score"].mean():.2f}")
    print("-" * 50)


logistic_regression
Train_score: 0.63
Test_score:  0.63
--------------------------------------------------
knn_classifier
Train_score: 0.80
Test_score:  0.65
--------------------------------------------------
decision_tree
Train_score: 1.00
Test_score:  0.73
--------------------------------------------------
random_forest
Train_score: 1.00
Test_score:  0.82
--------------------------------------------------
xgboost_classifier
Train_score: 0.89
Test_score:  0.84
--------------------------------------------------
catboost_classifier
Train_score: 0.87
Test_score:  0.84
--------------------------------------------------
lightgbm_classifier
Train_score: 0.84
Test_score:  0.83
--------------------------------------------------


# RandomizedSearch 

In [48]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    "model__depth": [4, 6, 8],                         
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__iterations": [500, 1000, 1500],                
    "model__l2_leaf_reg": [1, 3, 5, 7, 9],                 
    "model__random_strength": [0, 1, 5, 10],
    "model__auto_class_weights": ["None", "Balanced", "SqrtBalanced"],}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cat_pipeline=Pipeline(steps=[("preprocessing",preprocessing),
                               ("model",CatBoostClassifier( verbose=0, random_state=42,thread_count=8))])


search=RandomizedSearchCV(cat_pipeline,param_grid,n_iter=20,cv=cv,scoring="f1",return_train_score=True,n_jobs=1,random_state=42)
search.fit(x,y)

print(f"Best_param: {search.best_params_}")
print(f"Best_score: {search.best_score_*100:.2f}")

Best_param: {'model__random_strength': 5, 'model__learning_rate': 0.05, 'model__l2_leaf_reg': 1, 'model__iterations': 1000, 'model__depth': 6, 'model__auto_class_weights': 'None'}
Best_score: 84.20


# Models_score with optuna

In [49]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    depth = trial.suggest_int("depth", 4, 8)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2)
    iterations = trial.suggest_int("iterations", 500, 1500)
    l2_leaf_reg = trial.suggest_int("l2_leaf_reg", 1, 10)
    auto_class_weights = trial.suggest_categorical("auto_class_weights", ["None", "Balanced", "SqrtBalanced"])

    model = CatBoostClassifier(depth=depth, learning_rate=learning_rate, iterations=iterations,
                               l2_leaf_reg=l2_leaf_reg, auto_class_weights=auto_class_weights,
                               verbose=0, random_state=42, thread_count=8)

    model_pipeline = Pipeline(steps=[("preprocessing", preprocessing), ("model", model)])

    score = cross_val_score(model_pipeline, x, y, cv=cv, scoring="f1", n_jobs=1)
    return score.mean()
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

print(f"Best_param: {study.best_params}")
print(f"Best_score: {study.best_value*100:.2f}")

[I 2026-11-10 14:46:53,296] A new study created in memory with name: no-name-6bbeb898-3dd2-4238-b232-03a11aaeb97d
[I 2026-11-10 14:48:29,914] Trial 0 finished with value: 0.8449888594916729 and parameters: {'depth': 5, 'learning_rate': 0.19063571821788408, 'iterations': 1232, 'l2_leaf_reg': 6, 'auto_class_weights': 'None'}. Best is trial 0 with value: 0.8449888594916729.
[I 2026-11-10 14:49:35,193] Trial 1 finished with value: 0.8442203060550145 and parameters: {'depth': 8, 'learning_rate': 0.12421185223120967, 'iterations': 1208, 'l2_leaf_reg': 1, 'auto_class_weights': 'None'}. Best is trial 0 with value: 0.8449888594916729.
[I 2026-11-10 14:50:12,679] Trial 2 finished with value: 0.8344105671222819 and parameters: {'depth': 4, 'learning_rate': 0.044846856872152424, 'iterations': 804, 'l2_leaf_reg': 6, 'auto_class_weights': 'SqrtBalanced'}. Best is trial 0 with value: 0.8449888594916729.
[I 2026-11-10 14:50:51,209] Trial 3 finished with value: 0.8384351605382939 and parameters: {'dept

Best_param: {'depth': 5, 'learning_rate': 0.14682157857515898, 'iterations': 1489, 'l2_leaf_reg': 4, 'auto_class_weights': 'None'}
Best_score: 84.68


# Best_model

In [50]:
cat_boost_pipeline=Pipeline(steps=[("preprocessing",preprocessing),
                                   ("model",CatBoostClassifier
                                   (depth=5, learning_rate=0.14682157857515898,
                                    iterations=1489, l2_leaf_reg=4,
                                    auto_class_weights="None",
                                    verbose=0, random_state=42,thread_count=8 ))])
cat_boost_pipeline.fit(x,y)
result=cross_validate(cat_boost_pipeline,x,y,cv=cv,scoring="f1",return_train_score=True,n_jobs=1)

print(f"Train_score: {result['train_score'].mean()*100:.2f}")
print(f"Test_score:  {result['test_score'].mean()*100:.2f}")


Train_score: 93.93
Test_score:  84.68


In [51]:
print((result["test_score"]*100).round(2))

[83.28 86.72 83.91 84.43 85.04]


# Confusion Matrix & Classification Report

In [52]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

y_pred = cross_val_predict(cat_boost_pipeline, x, y, cv=cv, n_jobs=1)

In [53]:
print(classification_report(y, y_pred, target_names=["Paid", "Failed to Pay"]))

               precision    recall  f1-score   support

         Paid       0.94      0.99      0.96     25321
Failed to Pay       0.95      0.76      0.85      7088

     accuracy                           0.94     32409
    macro avg       0.95      0.88      0.90     32409
 weighted avg       0.94      0.94      0.94     32409



In [54]:
cm = confusion_matrix(y, y_pred)
fig_4 = px.imshow(cm, text_auto=True,color_continuous_scale="Purples",
                  x=["Paid", "Failed to Pay"], y=["Paid", "Failed to Pay"],
                  labels={"x": "Predicted", "y": "Actual"}, title="Confusion Matrix")
fig_4.show()

The model is very precise: when it says a borrower will fail to pay, it is right 95% of the time. Its recall is 76%, so it catches 3 out of 4 borrowers who fail to pay. The main weakness is the 1,691 borrowers it missed.

# Threshold

In [55]:
from sklearn.metrics import precision_score, recall_score

y_proba = cross_val_predict(cat_boost_pipeline, x, y, cv=cv, method="predict_proba", n_jobs=1)[:, 1]

for t in [0.5, 0.3, 0.2, 0.1]:
    pred = (y_proba >= t).astype(int)
    print(f"Threshold {t}: Precision {precision_score(y, pred)*100:.1f}%  Recall {recall_score(y, pred)*100:.1f}%")

Threshold 0.5: Precision 95.4%  Recall 76.1%
Threshold 0.3: Precision 87.6%  Recall 80.3%
Threshold 0.2: Precision 79.3%  Recall 83.6%
Threshold 0.1: Precision 63.0%  Recall 89.1%


Lowering the threshold catches more borrowers who fail to pay (higher recall), but rejects more good borrowers (lower precision). We kept 0.5 because it gives the best F1, but a bank should choose the threshold based on which mistake costs more.


### Effect of loan_to_income

| Features | Test F1 |
|---|---|
| Without loan_to_income | 84.25 |
| With loan_to_income| 84.68 |

In [56]:
importance = pd.DataFrame({"feature": cat_boost_pipeline["preprocessing"].get_feature_names_out(),
                           "importance": cat_boost_pipeline["model"].feature_importances_}).sort_values("importance",ascending=False)
importance

,feature,importance
1,person_income,16.064705
7,loan_to_income,12.696519
17,loan_grade,12.560999
10,person_home_ownership_RENT,12.189926
5,loan_percent_income,11.711055
2,person_emp_length,5.908403
4,loan_int_rate,5.793780
9,person_home_ownership_OWN,4.952784
3,loan_amnt,3.422121
0,person_age,3.261821


# Conclusion
- We compared 7 models with 5-fold cross-validation using F1, because the data is imbalanced (78% / 22%).
- CatBoost was the best model. SMOTE and BorderlineSMOTE did not improve the results.
- After tuning with RandomizedSearchCV and Optuna, the final CatBoost reached a test F1 of 84.68.
- The most important features are person_income, loan_to_income, loan_grade and home ownership (RENT).
- The model is saved with joblib and used in a Streamlit app.
- Precision for "Failed to Pay" is 95% and recall is 76%: the model rarely flags a good borrower, but misses about 1 in 4 bad ones.
- The model has some overfitting (train 93.9 vs test 84.7), but the test score is stable across the 5 folds.

In [57]:
joblib.dump(cat_boost_pipeline,"catboost_model.pkl")

['catboost_model.pkl']

In [58]:
%%writefile loan.py
import pandas as pd
import streamlit as st
import joblib

st.set_page_config(layout='wide', page_title='loan project')

st.title('Loan Status Prediction')
st.write('Enter the borrower and loan details, then press **Predict** to see if the borrower is likely to repay.')

df = pd.read_csv('credit_risk_clean.csv')

ml_model = joblib.load('catboost_model.pkl')

with st.expander('Show sample of the data'):
    st.dataframe(df.head())

st.sidebar.header('Borrower Info')

age = st.sidebar.number_input('Age', min_value=20, max_value=94, value=25, step=1)

income = st.sidebar.number_input('Annual Income', min_value=4000, max_value=2000000, value=50000, step=1000)

emp_length = st.sidebar.number_input('Employment Length (years)', min_value=0, max_value=41, value=3, step=1)

home_ownership = st.sidebar.selectbox('Home Ownership', df['person_home_ownership'].unique())

cred_hist_length = st.sidebar.number_input('Credit History Length (years)', min_value=2, max_value=30, value=4, step=1)

default_on_file = st.sidebar.selectbox('Previous Default', ['N', 'Y'])
st.sidebar.caption('N = never failed to pay a loan before')
st.sidebar.caption('Y = failed to pay a loan before')

col1, col2 = st.columns(2)

with col1:
    loan_amnt = st.number_input('Loan Amount', min_value=500, max_value=35000, value=10000, step=500)
    loan_intent = st.selectbox('Loan Intent', df['loan_intent'].unique())

with col2:
    loan_grade = st.selectbox('Loan Grade', ['A', 'B', 'C', 'D', 'E', 'F', 'G'])
    int_rate = st.number_input('Interest Rate (%)', min_value=5.0, max_value=25.0, value=11.0, step=0.1)

loan_to_income = loan_amnt / income
loan_percent_income = round(loan_to_income, 2)

st.metric('Loan to Income', f'{loan_to_income:.0%}')

if st.button('Predict', type='primary'):
    if emp_length > age - 14 or cred_hist_length > age - 14:
        st.warning('Employment length and credit history must fit the borrower age.')
        st.stop()

    if loan_to_income > 0.83:
        st.warning('The loan is too large compared to the income for this model.')
        st.stop()

    new_data = pd.DataFrame(columns=df.columns.drop('loan_status'),
                            data=[[age, income, home_ownership, emp_length, loan_intent, loan_grade, loan_amnt,
                                   int_rate, loan_percent_income, default_on_file, cred_hist_length, loan_to_income]])

    prediction = ml_model.predict(new_data)[0]
    repay_chance = ml_model.predict_proba(new_data)[0][0]

    if prediction == 0:
        st.success(f'The borrower is likely to repay the loan ({repay_chance:.1%} chance to repay)')
    else:
        st.error(f'The borrower is unlikely to repay the loan ({repay_chance:.1%} chance to repay)')


Writing loan.py


In [59]:
! streamlit run loan.py

^C
